Homework 11
==========
<link rel="stylesheet" href="/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/02_Script/02_1_Function/styles.css">

> 📌 1.	Formulate an optimization model (a linear program) to find the cheapest diet that satisfies the maximum and minimum daily nutrition constraints, and solve it using PuLP.  Turn in your code and the solution. (The optimal solution should be a diet of air-popped popcorn, poached eggs, oranges, raw iceberg lettuce, raw celery, and frozen broccoli. UGH!)</br>

</br>
</br>

In [9]:
import pandas as pd
from pulp import *


In [11]:
# Correct assignment of file paths
diet_large = "/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/01_Data/Homework11_ISYE6501/data 15.2/diet_large.xls"
diet = "/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/01_Data/Homework11_ISYE6501/data 15.2/diet.xls"

# Load the readxl package to read the Excel file

diet_large_df = pd.read_excel(diet_large, sheet_name="Sheet1", header=1)
diet_df = pd.read_excel(diet, sheet_name="Sheet1")


In [68]:
food=diet_df.loc[0:63,"Foods"].tolist()

nutrients=diet_df.columns.to_list()[3:]

cost=dict(zip(food, diet_df.loc[0:63,"Price/ Serving"].tolist()))


In [ ]:
a = {j: dict(zip(food, diet_df[j])) for j in nutrients}


min_val=diet_df.iloc[65,3:].tolist()
max_val=diet_df.iloc[66,3:].tolist()



L_MIN= { j: min_val[i] for i, j in enumerate(nutrients) }
U_MAX={ j: max_val[i] for i, j in enumerate(nutrients) }




In [79]:


Problem1 = LpProblem("Diet Problem", LpMinimize)
x = LpVariable.dicts("x", food, lowBound=0)

Problem1 += lpSum(cost[i] * x[i] for i in food)

for j in nutrients:
    Problem1 += lpSum(a[j][i] * x[i] for i in food) >= L_MIN[j]
    Problem1 += lpSum(a[j][i] * x[i] for i in food) <= U_MAX[j]

Problem1.solve()
for v in Problem1.variables():
    if v.varValue > 1e-6:
        print(v.name, v.varValue)
print("Total cost =", value(Problem1.objective))

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/6_/4v4px4w941x057x3xr2z3sb00000gn/T/80a3df5c4d53448fb13ca6d6e04e135a-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/6_/4v4px4w941x057x3xr2z3sb00000gn/T/80a3df5c4d53448fb13ca6d6e04e135a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 27 COLUMNS
At line 1286 RHS
At line 1309 BOUNDS
At line 1310 ENDATA
Problem MODEL has 22 rows, 64 columns and 1194 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 22 (0) rows, 64 (0) columns and 1194 (0) elements
0  Obj 0 Primal inf 21.63092 (11)
9  Obj 4.3371168
Optimal - objective value 4.3371168
Optimal objective 4.33711681 - 9 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU 

<link rel="stylesheet" href="/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/02_Script/02_1_Function/styles.css">

> 📌 2.	Please add to your model the following constraints (which might require adding more variables) and solve the new model:
>* a.	If a food is selected, then a minimum of 1/10 serving must be chosen. (Hint: now you will need two variables for each food i: whether it is chosen, and how much is part of the diet. You’ll also need to write a constraint to link them.)
>* b.	Many people dislike celery and frozen broccoli. So at most one, but not both, can be selected.
>* c.	To get day-to-day variety in protein, at least 3 kinds of meat/poultry/fish/eggs must be selected. [If something is ambiguous (e.g., should bean-and-bacon soup be considered meat?), just call it whatever you think is appropriate – I want you to learn how to write this type of constraint, but I don’t really care whether we agree on how to classify foods!]
</br>

In [ ]:
Upper_bound =100

Problem2 = LpProblem("Diet Problem with constraints", LpMinimize)

x = LpVariable.dicts("Servings", food, lowBound=0)
y = LpVariable.dicts("Chosen", food, lowBound=0, upBound=1, cat="Binary")

Problem2+= lpSum(cost[i] * x[i] for i in food)



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pulp/pulp.py:1489: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pulp/pulp.py:1865: UserWarning: Overwriting previously set objective.
  warnings.warn("Overwriting previously set objective.")


In [83]:
# Contrains 1. the servings of each food to be 0 if not chosen and at least 0.1 if chosen, with an upper bound of 100 servings

for i in food:
    Problem2 += x[i] <= Upper_bound * y[i]
    Problem2 += x[i] >= 0.1 * y[i]

# Constraints 2. the total number of servings of all foods to be at most 15

for j in nutrients:
    Problem2 += lpSum(a[j][i] * x[i] for i in food) >= L_MIN[j]
    Problem2 += lpSum(a[j][i] * x[i] for i in food) <= U_MAX[j]

#constraint 3. the total number of servings of "Celery, Raw" and "Frozen Broccoli" to be at most 1

Problem2 += y["Celery, Raw"] + x["Frozen Broccoli"] <=1

# constraint 4. selection of meats that can be chosen, at least 3 of the 6 meat options must be selected

meat_foods_selection=['Roasted Chicken',"Scrambled Eggs","Bologna,Turkey","Frankfurter, Beef","Pork","Hotdog, Plain"]

Problem2 += lpSum(y[i] for i in meat_foods_selection) >=3


In [84]:
# solution problem 2 

Problem2.solve()
for v in Problem2.variables():
    if v.varValue > 1e-6:
        print(v.name, v.varValue)
print("Total cost =", value(Problem2.objective))

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/6_/4v4px4w941x057x3xr2z3sb00000gn/T/81a115b5c2a345b29be03f1a7ea02367-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/6_/4v4px4w941x057x3xr2z3sb00000gn/T/81a115b5c2a345b29be03f1a7ea02367-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 307 COLUMNS
At line 3346 RHS
At line 3649 BOUNDS
At line 3714 ENDATA
Problem MODEL has 302 rows, 128 columns and 2908 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0.0357797 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 61 strengthened rows, 0 substitutions
Cgl0004I processed model has 141 rows, 128 columns (64 integer (64 of which binary)) and 861 elements
Cbc0038I Initial state - 8 integers